In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Learning")
    .master("local[*]")
    .getOrCreate()
)

df = spark.createDataFrame(
    [("Alice", 30), ("Bob", 25)],
    ["name", "age"]
)

df.show()

spark.stop()

In [ ]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster("local[*]").setAppName("RatingsHistogram")
sc = SparkContext(conf = conf)

lines = sc.textFile("ml-100k/u.data")
ratings = lines.map(lambda x: x.split()[2])
result = ratings.countByValue()

for rating, count in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(rating, count)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/10 11:13:11 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/10 11:13:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/10 11:13:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4 34174
3 27145
5 21201
2 11370
1 6110


In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster("local").setAppName("FakeFriends")
sc = SparkContext(conf = conf)

def parseLine(line):
    fields = line.split(',')
    age = int(fields[2])
    numFriends = int(fields[3])
    return (age, numFriends)

lines = sc.textFile("./fakefriends.csv")
rdd = lines.map(parseLine)


# Collect data from the RDD
data = rdd.collect()

# Create table
table = PrettyTable()
table.field_names = ["Age", "Number of Friends"]

for age, num_friends in data:
    table.add_row([age, num_friends])
print("Number of Friends by Age")
print(table)


# Count up sum of friends and number of entries per age
# ((22, 200), 1)
# ((22, 300), 1)
totalsByAge = (
    rdd.mapValues(lambda x: (x, 1))
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
    )
# this RDD isn't being used for anything since we caculate it manually later
# alternatively, we could have joined on the two RDDs and then print from the cobined result 
averagesByAge = totalsByAge.mapValues(lambda x: x[0] / x[1])
totalsData = totalsByAge.collect()
totalsTable = PrettyTable()
totalsTable.field_names = ["Age", "Total Friends", "Count", "Average Friends"]

for age, (totalFriends, count) in sorted(totalsData):
    avgFriends = totalFriends / count
    totalsTable.add_row([age, totalFriends, count, avgFriends])

print("\nTotals by Age")
print(totalsTable)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/10 15:36:04 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/10 15:36:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/10 15:36:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Number of Friends by Age
+-----+-------------------+
| Age | Number of Friends |
+-----+-------------------+
|  33 |        385        |
|  33 |         2         |
|  55 |        221        |
|  40 |        465        |
|  68 |         21        |
+-----+-------------------+

Totals by Age
+-----+---------------+-------+-----------------+
| Age | Total Friends | Count | Average Friends |
+-----+---------------+-------+-----------------+
|  33 |      387      |   2   |      193.5      |
|  40 |      465      |   1   |      465.0      |
|  55 |      221      |   1   |      221.0      |
|  68 |       21      |   1   |       21.0      |
+-----+---------------+-------+-----------------+


In [1]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local').setAppName('MinTemps')
sc = SparkContext(conf = conf)

def parseLine(line):
    fields = line.split(',')
    stationId = fields[0]
    entryType = fields[2]
    temp = float(fields[3])
    return (stationId, entryType, temp)

lines = sc.textFile('./weatherData1800s.csv')
parsedLines = lines.map(parseLine)
maxTemps = parsedLines.filter(lambda x: 'TMAX' in x[1])
stationTemps = maxTemps.map(lambda x: (x[0], x[2]))
maxTemps = stationTemps.reduceByKey(lambda x, y: max(x,y))
results = maxTemps.collect()

table = PrettyTable()
table.field_names = ["Station ID", "Temp"]

for (id, temp) in results:
    table.add_row([id, temp])

print(table)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 10:11:23 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/11 10:11:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 10:11:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------+-------+
|  Station ID |  Temp |
+-------------+-------+
| ITE00100554 | 323.0 |
| EZE00100082 | 323.0 |
+-------------+-------+


In [1]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local').setAppName('WordCount')
sc = SparkContext(conf=conf)

input = sc.textFile('./pride_and_prejudice.txt')
words = input.flatMap(lambda x: x.split())
wordCounts = words.countByValue()

table = PrettyTable()
table.field_names = ['Word', 'Count']

for word, count in wordCounts.items():
    table.add_row([word, count])

print(table)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 10:30:40 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/11 10:30:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 10:30:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------------------------------+-------+
|               Word               | Count |
+----------------------------------+-------+
|               The                |  285  |
|             Project              |   79  |
|            Gutenberg             |   60  |
|              eBook               |   6   |
|                of                |  3899 |
|              Pride               |   7   |
|               and                |  3444 |
|            Prejudice             |   5   |
|               This               |   49  |
|                is                |  861  |
|               for                |  1041 |
|               the                |  4509 |
|               use                |   23  |
|              anyone              |   20  |
|             anywhere             |   3   |
|                in                |  1923 |
|              United              |   15  |
|              States              |   7   |
|               most               |  196  |
|         

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable
import re

conf = SparkConf().setMaster('local').setAppName('WordCount')
sc = SparkContext(conf=conf)

# \b = word boundary
# \w+ = one or more word characters
# (?:) = non-capture groups ignore what matches after the ':' 
# * = zero or more 
WORD_RE = re.compile(r"\b\w+(?:'\w+)*\b", re.UNICODE)

def normalizeWords(text):
    return WORD_RE.findall(text.lower())
    # Doing it this way will split contractions still
    # e.g. it's --> it, s
    # return re.compile(r'\W+', re.UNICODE).split(text.lower())

input = sc.textFile('./pride_and_prejudice.txt')
words = input.flatMap(normalizeWords)
wordCounts = words.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)
wordCountsSorted = wordCounts.sortBy(lambda x: x[1], ascending=False)
# we could have also flipped the key/value and sorted like this:
# wordCountsSorted = wordCounts.map(lambda x, y: (y, x)).sortByKey()

table = PrettyTable()
table.field_names = ['Word', 'Count']

for word, count in wordCountsSorted.collect():
        table.add_row([word, count])

print(table)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 12:34:39 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/11 12:34:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 12:34:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------------+-------+
|        Word       | Count |
+-------------------+-------+
|        the        |  4846 |
|         to        |  4405 |
|         of        |  3962 |
|        and        |  3835 |
|        her        |  2260 |
|         i         |  2098 |
|         a         |  2094 |
|         in        |  2051 |
|        was        |  1874 |
|        she        |  1732 |
|        that       |  1620 |
|         it        |  1603 |
|        not        |  1520 |
|        you        |  1417 |
|         he        |  1350 |
|        his        |  1289 |
|         be        |  1280 |
|         as        |  1239 |
|        had        |  1181 |
|        with       |  1149 |
|        for        |  1118 |
|        but        |  1040 |
|         is        |  945  |
|        have       |  883  |
|         at        |  831  |
|         mr        |  807  |
|        him        |  765  |
|         on        |  745  |
|         by        |  724  |
|         my        |  710  |
|         

In [1]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local').setAppName('MinTemps')
sc = SparkContext(conf = conf)

def parseLine(line):
    fields = line.split(',')
    id = fields[0]
    orderPrice = float(fields[2])
    return (id, orderPrice)

lines = sc.textFile('./customer-orders.csv')
parsedLines = lines.map(parseLine)
customerTotal = parsedLines.reduceByKey(lambda x, y: x + y).sortBy(lambda x: x[1], ascending=False)

table = PrettyTable()
table.field_names = ["ID", "Total"]

for (id, total) in customerTotal.collect():
    table.add_row([id, total])

print(table)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 12:51:12 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/11 12:51:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 12:51:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----+--------------------+
| ID |       Total        |
+----+--------------------+
| 68 | 6375.449999999997  |
| 73 | 6206.199999999999  |
| 39 | 6193.109999999999  |
| 54 | 6065.389999999999  |
| 71 | 5995.660000000003  |
| 2  |      5994.59       |
| 97 | 5977.189999999995  |
| 46 | 5963.109999999999  |
| 42 | 5696.840000000003  |
| 59 |      5642.89       |
| 41 |      5637.62       |
| 0  | 5524.949999999998  |
| 8  | 5517.240000000001  |
| 85 |      5503.43       |
| 61 | 5497.479999999998  |
| 32 | 5496.050000000004  |
| 58 | 5437.7300000000005 |
| 63 | 5415.150000000001  |
| 15 | 5413.510000000001  |
| 6  | 5397.879999999998  |
| 92 | 5379.280000000002  |
| 43 |      5368.83       |
| 70 | 5368.249999999999  |
| 72 |      5337.44       |
| 34 |       5330.8       |
| 9  | 5322.649999999999  |
| 55 | 5298.090000000002  |
| 90 | 5290.409999999998  |
| 64 | 5288.689999999996  |
| 93 | 5265.750000000001  |
| 24 | 5259.920000000003  |
| 33 | 5254.659999999998  |
| 62 | 5253.32000000

In [1]:
from pyspark.sql import SparkSession
from prettytable import PrettyTable

spark = SparkSession.builder \
    .appName("CustomerTotals") \
    .master("local") \
    .getOrCreate()

# Read CSV
df = spark.read.csv(
    "./customer-orders.csv",
    schema="customer_id STRING, item_id STRING, order_price DOUBLE"
)

# Create temporary SQL view
df.createOrReplaceTempView("orders")

# SQL query
customerTotal = spark.sql("""
    SELECT
        customer_id,
        SUM(order_price) AS total
    FROM orders
    GROUP BY customer_id
    ORDER BY total DESC
""")

# Display with PrettyTable
table = PrettyTable()
table.field_names = ["ID", "Total"]

for row in customerTotal.collect():
    table.add_row([row.customer_id, row.total])

print(table)

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/16 21:01:00 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/16 21:01:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 21:01:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----+--------------------+
| ID |       Total        |
+----+--------------------+
| 68 | 6375.449999999997  |
| 73 | 6206.199999999999  |
| 39 | 6193.109999999999  |
| 54 | 6065.389999999999  |
| 71 | 5995.660000000003  |
| 2  |      5994.59       |
| 97 | 5977.189999999995  |
| 46 | 5963.109999999999  |
| 42 | 5696.840000000003  |
| 59 |      5642.89       |
| 41 |      5637.62       |
| 0  | 5524.949999999998  |
| 8  | 5517.240000000001  |
| 85 |      5503.43       |
| 61 | 5497.479999999998  |
| 32 | 5496.050000000004  |
| 58 | 5437.7300000000005 |
| 63 | 5415.150000000001  |
| 15 | 5413.510000000001  |
| 6  | 5397.879999999998  |
| 92 | 5379.280000000002  |
| 43 |      5368.83       |
| 70 | 5368.249999999999  |
| 72 |      5337.44       |
| 34 |       5330.8       |
| 9  | 5322.649999999999  |
| 55 | 5298.090000000002  |
| 90 | 5290.409999999998  |
| 64 | 5288.689999999996  |
| 93 | 5265.750000000001  |
| 24 | 5259.920000000003  |
| 33 | 5254.659999999998  |
| 62 | 5253.32000000

In [2]:
from pyspark.sql import SparkSession, Row

spark = SparkSession.builder.appName("SparkSQL").getOrCreate()

def mapper(line):
    fields = line.split(',')
    return Row( ID=int(fields[0]), name=str(fields[1].encode('utf-8')), age=int(fields[2]), num_friends=int(fields[3]) )

lines = spark.sparkContext.textFile('./fakefriends.csv')
people = lines.map(mapper)

# building Dataframe from RDD (one possible pattern)
schemaPeople = spark.createDataFrame(people).cache()
schemaPeople.createOrReplaceTempView('people')

adults = spark.sql("SELECT * FROM people WHERE age >= 18")

for a in adults.collect():
    print(a)

schemaPeople.groupBy('age').count().orderBy('age').show()
spark.stop()

Row(ID=0, name="b'Will'", age=33, num_friends=385)
Row(ID=1, name="b'Jean-Luc'", age=33, num_friends=2)
Row(ID=2, name="b'Hugh'", age=55, num_friends=221)
Row(ID=3, name="b'Deanna'", age=40, num_friends=465)
Row(ID=4, name="b'Quark'", age=68, num_friends=21)
+---+-----+
|age|count|
+---+-----+
| 17|    1|
| 33|    2|
| 40|    1|
| 55|    1|
| 68|    1|
+---+-----+



In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SparkSQL').getOrCreate()

people = spark.read.option('header', True).option('inferschema', True)\
.csv('./fakefriends-header.csv')

print('Here is our inferred schema')
people.printSchema()

people.select('name').show()

people.filter(people.age <= 35).show()

people.groupBy('age').count().show()

people.select(people.name, people.age + 10).show()

spark.stop()

Here is our inferred schema
root
 |-- userID: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- friends: integer (nullable = true)

+--------+
|    name|
+--------+
|    Will|
|Jean-Luc|
|    Hugh|
|  Deanna|
|   Quark|
|  Weyoun|
|  Gowron|
|    Will|
|  Jadzia|
|    Hugh|
|     Odo|
|     Ben|
|   Keiko|
|Jean-Luc|
|    Hugh|
|     Rom|
|  Weyoun|
|     Odo|
|Jean-Luc|
|  Geordi|
+--------+
only showing top 20 rows
+------+--------+---+-------+
|userID|    name|age|friends|
+------+--------+---+-------+
|     0|    Will| 33|    385|
|     1|Jean-Luc| 26|      2|
|     9|    Hugh| 27|    181|
|    16|  Weyoun| 22|    323|
|    17|     Odo| 35|     13|
|    21|   Miles| 19|    268|
|    22|   Quark| 30|     72|
|    24|  Julian| 25|      1|
|    25|     Ben| 21|    445|
|    26|  Julian| 22|    100|
|    32|     Nog| 26|    281|
|    35| Beverly| 27|    305|
|    36|  Kasidy| 32|     81|
|    39|    Morn| 31|    192|
|    44|   Nerys

In [3]:
from pyspark.sql import SparkSession, functions as func

spark = SparkSession.builder.appName('SparkSQL').getOrCreate()

people = spark.read.option('header', True).option('inferschema', True)\
.csv('./fakefriends-header.csv')

print('Here is our inferred schema')
people.printSchema()

people.select('name', 'age').show()

people.groupBy('age').avg('friends').sort('age').show()

people.groupBy('age').agg(func.round(func.avg('friends'), 2)).sort('age').show()

people.groupBy('age').agg(func.round(func.avg('friends'), 2)
                          .alias('friends_avg')).sort('age').show()

spark.stop()

Here is our inferred schema
root
 |-- userID: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- friends: integer (nullable = true)

+--------+---+
|    name|age|
+--------+---+
|    Will| 33|
|Jean-Luc| 26|
|    Hugh| 55|
|  Deanna| 40|
|   Quark| 68|
|  Weyoun| 59|
|  Gowron| 37|
|    Will| 54|
|  Jadzia| 38|
|    Hugh| 27|
|     Odo| 53|
|     Ben| 57|
|   Keiko| 54|
|Jean-Luc| 56|
|    Hugh| 43|
|     Rom| 36|
|  Weyoun| 22|
|     Odo| 35|
|Jean-Luc| 45|
|  Geordi| 60|
+--------+---+
only showing top 20 rows
+---+------------------+
|age|      avg(friends)|
+---+------------------+
| 18|           343.375|
| 19|213.27272727272728|
| 20|             165.0|
| 21|           350.875|
| 22|206.42857142857142|
| 23|             246.3|
| 24|             233.8|
| 25|197.45454545454547|
| 26|242.05882352941177|
| 27|           228.125|
| 28|             209.1|
| 29|215.91666666666666|
| 30| 235.8181818181818|
| 31|            267.25|
| 32|

In [1]:
from pyspark.sql import SparkSession, functions as func

spark = SparkSession.builder.appName('WordCount').getOrCreate()

inputDF = spark.read.text('./pride_and_prejudice.txt')

words = inputDF.select(func.explode(func.split(inputDF.value, r'\W+')).alias('word'))
wordsWithoutEmtptyString = words.filter(words.word != '')

lowercaseWords = wordsWithoutEmtptyString.select(func.lower(wordsWithoutEmtptyString.word).alias('word'))

wordCounts = lowercaseWords.groupBy('word').count()

wordCountsSorted = wordCounts.sort('count')

wordCountsSorted.show(wordCountsSorted.count())

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 14:48:26 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/17 14:48:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 14:48:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------------+-----+
|             word|count|
+-----------------+-----+
|       palpitated|    1|
|           _what_|    1|
|        imitation|    1|
|          solaced|    1|
|     premeditated|    1|
|          elevate|    1|
|       lieutenant|    1|
|    gratification|    1|
|            pools|    1|
|           coaxed|    1|
|         singling|    1|
|      unavoidably|    1|
|      irreligious|    1|
|       superseded|    1|
|       concluding|    1|
|           biting|    1|
|         tortured|    1|
|      ingratitude|    1|
|          painted|    1|
|      laboriously|    1|
|       productive|    1|
|            oddly|    1|
|        reverting|    1|
|         imputing|    1|
|         noticing|    1|
|        surveying|    1|
|        construed|    1|
|           filled|    1|
|    irretrievable|    1|
|           sequel|    1|
|            scorn|    1|
|      distributor|    1|
|            staff|    1|
|          memling|    1|
|          achieve|    1|
|        _di

In [ ]:
from pyspark.sql import SparkSession, functions as func 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder.appName('Temps').getOrCreate()

schema = StructType([
    StructField('stationId', StringType(), True),
    StructField('date', IntegerType(), True),
    StructField('type', StringType(), True),
    StructField('temp', FloatType(), True)
])

df = spark.read.schema(schema).csv('./weatherData1800s.csv')
df.printSchema()


minTemps = df.filter(df.type == 'TMIN')

stationTemps = minTemps.select('stationId', 'temp')

minTempByStation = stationTemps.groupBy('stationId').min('temp')
minTempByStation.show()

# withColumn returns a new DataFrame by adding a column or replacing the existing column that has the same name.
minTempByStationF = minTempByStation.withColumn('temp',
                                                func.round(func.col('min(temp)') * 0.1 * (9.0 / 5.0) + 32.0, 2))\
                                                .select('stationId', 'temp').sort('temp')

minTempByStationF.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 15:52:31 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/17 15:52:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 15:52:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- stationId: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- temp: float (nullable = true)



+-----------+---------+
|  stationId|min(temp)|
+-----------+---------+
|ITE00100554|   -148.0|
|EZE00100082|   -135.0|
+-----------+---------+

+-----------+----+
|  stationId|temp|
+-----------+----+
|ITE00100554|5.36|
|EZE00100082| 7.7|
+-----------+----+



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql import functions as func

spark = SparkSession.builder.appName('CustomerSpending').getOrCreate()

schema = StructType([
    StructField('customerId', StringType(), True),
    StructField('orderId', StringType(), True),
    StructField('amount', FloatType(), True)
])

df = spark.read.schema(schema).csv('./customer-orders.csv')

df.groupBy('customerId') \
  .agg(func.sum('amount').alias('total_amount')) \
  .orderBy('total_amount', ascending=False) \
  .show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 18:15:55 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/17 18:15:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 18:15:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------+------------------+
|customerId|      total_amount|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows


In [1]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

@udf(returnType=StringType())
def user_details(name, age):
    return f"Name: {name}, Age: {age}"

spark = SparkSession.builder.appName('UDF_Example').getOrCreate()

people = spark.read.csv('./fakefriendsheader.csv', header=True)

people.withColumn(
    "name_age",
    user_details("name", "age")
).show()

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/22 12:14:22 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/22 12:14:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 12:14:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+--------+---+--------+--------------------+
| id|    name|age|#friends|            name_age|
+---+--------+---+--------+--------------------+
|  0|    Will| 33|     385| Name: Will, Age: 33|
|  1|Jean-Luc| 33|       2|Name: Jean-Luc, A...|
|  2|    Hugh| 55|     221| Name: Hugh, Age: 55|
|  3|  Deanna| 40|     465|Name: Deanna, Age...|
|  4|   Quark| 68|      21|Name: Quark, Age: 68|
|  5|  Wesley| 17|       1|Name: Wesley, Age...|
+---+--------+---+--------+--------------------+



In [1]:
from pyspark.sql import SparkSession, functions as func 
from pyspark.sql.types import StructType, StructField, IntegerType, LongType

spark = SparkSession.builder.appName("PopularMovies").getOrCreate()
spark.sparkContext.setLogLevel('WARN')

schema = StructType([ \
    StructField("UserId", IntegerType(), True), \
    StructField("MovieId", IntegerType(), True), \
    StructField("rating", IntegerType(), True), \
    StructField("timestamp", LongType(), True)])

moviesDF = spark.read.option('sep', '\t').schema(schema).csv('./ml-100k/u.data')
topMoviesIds = moviesDF.groupBy('UserId').count().orderBy(func.desc('count'))

topMoviesIds.show(10)

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/22 15:37:35 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/22 15:37:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 15:37:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+-----+
|UserId|count|
+------+-----+
|   405|  737|
|   655|  685|
|    13|  636|
|   450|  540|
|   276|  518|
|   416|  493|
|   537|  490|
|   303|  484|
|   234|  480|
|   393|  448|
+------+-----+
only showing top 10 rows


In [ ]:
# example from broadcast variables using join instead

from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, StringType

spark = SparkSession.builder.appName("PopularMovies").getOrCreate()
spark.sparkContext.setLogLevel('WARN')

ratingsSchema = StructType([ \
                     StructField("userID", IntegerType(), True), \
                     StructField("movieID", IntegerType(), True), \
                     StructField("rating", IntegerType(), True), \
                     StructField("timestamp", LongType(), True)])


moviesSchema = StructType([
    StructField("movieID", IntegerType(), True),
    StructField("title",  StringType(), True)
])

reviewsDF = spark.read.option("sep", "\t").schema(ratingsSchema).csv("./ml-100k/u.data")
moviesDF = spark.read.option("sep", "|").schema(moviesSchema).csv("./ml-100k/u.item")

movieCounts = reviewsDF.groupBy('movieID').count()

sortedMoviesWithNames = movieCounts.join(moviesDF, "movieID").orderBy(func.desc("count"))

sortedMoviesWithNames.show(10)

spark.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/22 16:20:07 WARN Utils: Your hostname, Rev-PF3CHFLT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/22 16:20:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 16:20:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+-----+--------------------+
|movieID|count|               title|
+-------+-----+--------------------+
|     50|  583|    Star Wars (1977)|
|    258|  509|      Contact (1997)|
|    100|  508|        Fargo (1996)|
|    181|  507|Return of the Jed...|
|    294|  485|    Liar Liar (1997)|
|    286|  481|English Patient, ...|
|    288|  478|       Scream (1996)|
|      1|  452|    Toy Story (1995)|
|    300|  431|Air Force One (1997)|
|    121|  429|Independence Day ...|
+-------+-----+--------------------+
only showing top 10 rows
